# Northstar router: fine-tuning run (Phase 5)

Runs the whole fine-tuning step on a free Kaggle GPU, about 45 minutes in total.

**Before you start (Kaggle settings, right panel):** Accelerator = **GPU T4 x1**, Internet = **On**.
Then **Run all**. No API key is needed in this notebook.

What it does:
1. Chooses the base model and learning rate using the 30 **validation** chats only.
2. Trains the chosen settings with 5 seeds, keeps the seed with the best validation score.
3. Scores the router once on the 40 **test** chats, through the same evaluation script as the prompted routers.
4. Zips the results for download (`finetune_results.zip`, a few hundred KB).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/Salikawtar/northstar-agent.git /kaggle/working/northstar-agent
%cd /kaggle/working/northstar-agent
!pip install -q -r requirements.txt -r requirements-finetune.txt && pip install -q -e .
!git log -1 --format='Code version: %h %s'

## Step 1: choose base model and learning rate (validation only, about 30 min)

In [ ]:
!python -m northstar.finetune.train_router select --quiet

## Step 2: final training, 5 seeds (about 12 min)

In [ ]:
!python -m northstar.finetune.train_router final --quiet --out models/finetuned_router

## Step 3: official test score, same evaluation as the prompted routers (runs on CPU, like production)

In [ ]:
!python -m northstar.agent.router --split test --model finetuned

## Step 4: learning curves for the report appendix, then zip the results

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, json
h = pd.read_csv("src/northstar/finetune/results/training_history.csv")
router_seed = json.load(open("src/northstar/finetune/results/training_log.json"))["router_seed"]
fig, ax = plt.subplots(figsize=(7, 4))
for seed, g in h.groupby("seed"):
    style = dict(lw=2.2) if seed == router_seed else dict(lw=0.8, alpha=0.4)
    ax.plot(g.epoch, g.train_loss, color="#1f77b4", **style)
    ax.plot(g.epoch, g.val_loss, color="#d62728", **style)
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_title("Training (blue) and validation (red) loss, 5 seeds; bold = router")
plt.tight_layout(); plt.savefig("src/northstar/finetune/results/learning_curves.png", dpi=150); plt.show()

In [ ]:
!cd /kaggle/working/northstar-agent && zip -qr /kaggle/working/finetune_results.zip src/northstar/finetune/results outputs/results/router_finetuned-* && ls -lh /kaggle/working/finetune_results.zip
print("Download finetune_results.zip from the Output panel (right side), then send it over.")

### Optional: keep the trained model
The checkpoint (`models/finetuned_router`, about 1.1 GB) is not needed for the report, because the test results are in the zip.
Run the cell below only if you also want to run the fine-tuned router on your laptop.

In [ ]:
# !cd /kaggle/working/northstar-agent && zip -qr /kaggle/working/finetuned_router_checkpoint.zip models/finetuned_router